In [ ]:
%pip install gdown tensorboard zarr earthaccess folium
%pip install terratorch==1.1.1

In [ ]:
from pathlib import Path
import os

import gdown

In [ ]:

hwds_google_drive_id = '1c5dQxAYfv4b4DCLyrjP0XyTERiofdHF0'
drive_url = f'https://drive.google.com/uc?id={hwds_google_drive_id}'
filename = '0095_S30'

if not Path(filename).exists():
  gdown.download(drive_url, f'{filename}.zip', quiet=False)
  !unzip {filename}.zip -d {filename}

In [ ]:
from typing import Any
from torchgeo.datasets import RasterDataset, stack_samples, unbind_samples
import matplotlib.pyplot as plt

class HWDSHLSDataset(RasterDataset):
    filename_glob = "*HLS.*.tif"
    filename_regex = r"^(?P<swathID>\d+)\.HLS\.(?P<sensor>S)30\.(?P<date>\d+)\.v(?P<version>\d+\.\d+)\.(?P<band>\w+)\.tif$"
    date_format = '%Y%j'

    separate_files = True
    is_image = True

    all_bands = ("B02", "B03", "B04", "B06", "B07", "B08", "EVENT", "Fmask")
    rgb_bands = ('B04', 'B03', 'B02')

    def plot(
            self,
            sample: dict[str, Any],
            show_titles: bool = True,
            suptitle: str | None = None,
        ):
        rgb_indices = []
        for band in self.rgb_bands:
            rgb_indices.append(self.bands.index(band))

        image = sample['image'][rgb_indices].permute(1, 2, 0)
        # DN = 10000 * REFLECTANCE
        # https://docs.sentinel-hub.com/api/latest/data/sentinel-2-l2a/
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))

        ax.imshow(image)
        ax.axis('off')

        if show_titles:
            ax.set_title('Image')

        if suptitle is not None:
            plt.suptitle(suptitle)

        return fig


This is the recommended approach for setting up the data and the labels

https://torchgeo.readthedocs.io/en/stable/api/datasets.html#torchgeo.datasets.RasterDataset.is_image

In [ ]:
from torch.utils.data import DataLoader
from torchgeo.samplers import RandomGeoSampler

patch_size = 224

data = HWDSHLSDataset(paths=filename, bands=("B02", "B03", "B04", "B06", "B07", "B08"))
label = HWDSHLSDataset(paths=filename, bands=['EVENT'])
label.is_image = False

dataset = data & label

sampler = RandomGeoSampler(
    dataset,
    size=patch_size,
    length=10,
)

dataloader = DataLoader(
    dataset,
    sampler=sampler,
    batch_size=2,
    collate_fn=stack_samples,
)

In [ ]:

print(len(dataloader))

for batch in dataloader:
    batch_shape = batch['image'].shape # [B, C, W, H]
    print(f"Batch: {batch.keys()}, Image Shape: {batch_shape}")

    sample = unbind_samples(batch)[0]
    sample_shape = sample['image'].shape # [C, W, H]
    print(f"Single Sample: {sample.keys()}")
    print(f"Image Shape: {sample_shape}")
    print(f"Mask Shape: {sample['mask'].shape}")

    # train_model(batch)
    break